# PyTorch: Custom Dataset Classification

In [ ]:
import torch
from torchinfo import summary
from torch import nn
from torch import optim
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from torchmetrics.classification import MulticlassAccuracy, MulticlassConfusionMatrix
from tqdm.auto import tqdm
from common import CV_DATASETS_DIR
import common.torch as ct

In [ ]:
ct.set_default_seed()
ct.set_default_optimizations()
device = ct.get_optimal_device()

In [ ]:
print(f"PyTorch: version {torch.__version__}")
print(f"PyTorch: {device.type.upper()} device")

In [ ]:
# Hyperparameters
BATCH_SIZE = 64
N_EPOCHS = 30
# Other parameters
IMAGE_SIZE = (64,64)

## Prepare Datasets

In [ ]:
DATASET_PATH = CV_DATASETS_DIR/"food"/"pizza_steak_sushi"

In [ ]:
TR_DATASET_DIR = DATASET_PATH / "train"
TS_DATASET_DIR = DATASET_PATH / "test"

In [ ]:
tr_dataset = datasets.ImageFolder(root=TR_DATASET_DIR, transform=v2.Compose([
    v2.Resize(size=IMAGE_SIZE),
    v2.TrivialAugmentWide(num_magnitude_bins=31),
    v2.ToImage(),
    v2.ToDtype(dtype=torch.float, scale=True)
]))
ts_dataset = datasets.ImageFolder(root=TS_DATASET_DIR, transform=v2.Compose([
    v2.Resize(size=IMAGE_SIZE),
    v2.ToImage(),
    v2.ToDtype(dtype=torch.float, scale=True)
]))
len(tr_dataset), len(ts_dataset)

In [ ]:
tr_dl = DataLoader(tr_dataset, batch_size=BATCH_SIZE, num_workers=2, shuffle=True)
ts_dl = DataLoader(ts_dataset, batch_size=BATCH_SIZE, num_workers=2, drop_last=True)
len(tr_dl), len(ts_dl)

In [ ]:
n_classes = len(tr_dataset.classes)
n_classes

## Define Model

In [ ]:
class TinyVggModel(nn.Module):
    def __init__(self, in_shape: int, out_shape: int, hidden_units: int):
        super().__init__()
        self.cv_block1 = nn.Sequential(
            nn.Conv2d(in_channels=in_shape, out_channels=hidden_units, kernel_size=3),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.cv_block2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hidden_units*13*13,
                      out_features=out_shape),
        )

    def forward(self, inputs):
        z = self.cv_block1(inputs)
        z = self.cv_block2(z)
        outputs = self.classifier(z)
        return outputs

In [ ]:
model = TinyVggModel(in_shape=3, out_shape=n_classes, hidden_units=32).to(device)

In [ ]:
summary(model, input_size=(64, 3)+IMAGE_SIZE)

## Train Model

In [ ]:
logs_dir, writer = ct.get_summary_writer("pt_transfer_learning", "efficientnet_b0", "5_epochs")
logs_dir

In [ ]:
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
accuracy = MulticlassAccuracy(num_classes=n_classes).to(device)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

In [ ]:
ct.train(model=model,
         tr_dl=tr_dl,
         ts_dl=ts_dl,
         optimizer=optimizer,
         criterion=criterion,
         metric=accuracy,
         n_epochs=N_EPOCHS,
         writer=writer,
         device=device,
         scheduler=scheduler)

## Evaluate Model

In [ ]:
def evaluate(model: nn.Module,
             loader: DataLoader,
             loss_fn: nn.Module,
             accuracy_fn: nn.Module,
             device: torch.device = device) -> dict:
    loss_avg = 0
    accu_avg = 0
    model.eval()
    preds = []
    truth = []
    with torch.inference_mode():
        for x, y_true in tqdm(loader):
            x, y_true = x.to(device), y_true.to(device)
            y_logits = model(x)
            y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1)
            loss_avg += loss_fn(y_logits, y_true).item()
            accu_avg += accuracy_fn(y_pred, y_true).item()
            preds.append(y_pred)
            truth.append(y_true)
        n_batches = len(loader)
        loss_avg /= n_batches
        accu_avg /= n_batches
    return {
        "truth": torch.flatten(torch.stack(truth)).cpu(),
        "preds": torch.flatten(torch.stack(preds)).cpu(),
        "loss": loss_avg,
        "accu": accu_avg,
    }

In [ ]:
results = evaluate(model, ts_dl, criterion, accuracy)
print(f"Loss: {results["loss"]}\nAccuracy: {results["accu"]}")

In [ ]:
metric = MulticlassConfusionMatrix(num_classes=n_classes)
metric.update(results["preds"], results["truth"])
metric.plot(labels=ts_dataset.classes);